# Inspecting a LoRA Adapter

This notebook loads a trained adapter and inspects it directly —
shapes, rank, parameter counts, and a visualization of the low-rank
decomposition. This is where LoRA stops being an abstract idea and
becomes concrete tensors you can look at.

In [ ]:
import torch
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM

ADAPTER_PATH = "../outputs/lora-run/final_adapter"

config = PeftConfig.from_pretrained(ADAPTER_PATH)
print(config)

## Load base model + adapter

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path, torch_dtype=torch.bfloat16
)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("Loaded.")

## List every LoRA layer and its A/B matrix shapes

Each LoRA layer has two small matrices: `lora_A` (shape `r x in_features`)
and `lora_B` (shape `out_features x r`). Their product `B @ A` approximates
the weight update `delta_W`, but is stored using far fewer parameters than
a full `out_features x in_features` matrix.

In [ ]:
for name, module in model.named_modules():
    if hasattr(module, "lora_A") and "default" in getattr(module, "lora_A", {}):
        A = module.lora_A["default"].weight
        B = module.lora_B["default"].weight
        full_equiv_params = A.shape[1] * B.shape[0]
        lora_params = A.numel() + B.numel()
        print(f"{name}")
        print(f"   lora_A: {tuple(A.shape)}  lora_B: {tuple(B.shape)}")
        print(f"   LoRA params: {lora_params:,}  vs full-matrix equivalent: {full_equiv_params:,}"
              f"  ({100*lora_params/full_equiv_params:.2f}% of full)")

## Visualize the magnitude of the learned update for one layer

Compute `delta_W = B @ A * (alpha / r)` for a single attention projection
and look at how large the update is relative to the original weight.

In [ ]:
import matplotlib.pyplot as plt

# Pick the first LoRA-wrapped module found
target_name, target_module = None, None
for name, module in model.named_modules():
    if hasattr(module, "lora_A") and "default" in getattr(module, "lora_A", {}):
        target_name, target_module = name, module
        break

A = target_module.lora_A["default"].weight.detach().float()
B = target_module.lora_B["default"].weight.detach().float()
scaling = target_module.scaling["default"]
delta_W = (B @ A) * scaling

print(f"Layer: {target_name}")
print(f"delta_W shape: {tuple(delta_W.shape)}")
print(f"delta_W mean abs value: {delta_W.abs().mean().item():.6f}")

plt.figure(figsize=(6, 4))
plt.hist(delta_W.flatten().numpy(), bins=100)
plt.title(f"Distribution of learned weight updates\n{target_name}")
plt.xlabel("delta_W value")
plt.ylabel("count")
plt.show()

## Exercise

- Compare `delta_W` magnitude across an early layer vs a late layer.
  Which layers did the adapter change more?
- Re-run this notebook after training with `r=64` instead of `r=16`.
  Does the distribution get wider?